In [ ]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


Question 1

In [ ]:
import numpy as np
from typing import Literal, Optional, Sequence, Union

#for defining attribute types for classification
AttributeType  = Literal["numeric", "binary", "nominal", "ordinal"]

def is_binary(arr: np.ndarray) -> bool:
    """for chekcing if an array contains only binary values (0s and 1s)."""
    if arr.dtype  == bool:
        return True
    return set(np.unique(arr)).issubset({0, 1})

def determine_attribute_type(arr: np.ndarray)  -> AttributeType:
    """
    to determinee the type of attribute:
    - 'nominal': strings or object arrays
    -  'binary': onyl has 0s and 1s or is boolean
    - 'numeric': int/float with more than two unique values
    
    - 'ordinal': not inferred; needs to be specified

    """
    if  np.issubdtype(arr.dtype, np.number):
        return "binary"  if is_binary(arr) else "numeric"
    return "nominal"

def minkowski_distance(x: np.ndarray, y: np.ndarray, p: Union[int, float]) -> float:
    """Calculate the Minkowski dis between the two vectors."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    if x.shape != y.shape:
        raise ValueError("Vectors need to have the same shape.")
    diff = np.abs(x - y)
    if p == np.inf:
        return  float(np.max(diff))
    return float(np.sum(diff ** p) ** (1.0 / p))

def binary_distance(x: np.ndarray, y: np.ndarray, method: Literal["hamming,""jaccard"] = "jaccard") -> float:
    """Compute the binary distance between two vectors."""
    x, y = np.asarray(x).astype(int), np.asarray(y).astype(int)
    if x.shape != y.shape:
        raise  ValueError("Binary vectors should  match in shape.")
    if method ==  "hamming":
        return np.mean(x != y)
    elif method == "jaccard":
        intersection = np.sum((x == 1) &  (y == 1))
        union = np.sum((x == 1) | (y == 1))
        return 0.0 if union == 0 else 1 - (intersection / union)
    else:
        raise ValueError("the Unknown binary distance method.")

def nominal_distance (x: np.ndarray, y: np.ndarray) -> float:
    """have the distance  be compute between nominal (categorical) values."""
    x, y = np.asarray(x), np.asarray(y)
    if x.shape != y.shape:
        raise ValueError("Nominal arrays must be the same length.")
    return  np.mean(x != y)

def ordinal_distance(x: np.ndarray, y: np.ndarray, order: Optional[Sequence] = None)  -> float:
    """have ordinal distance be compute based on ranking."""
    x, y = np.asarray(x),  np.asarray(y)
    if x.shape != y.shape:
        raise ValueError("Ordinal arrays must have the same shape.")
    if order is not None:
        rank_map = {value: i for i, value in enumerate(order)}
        xr, yr = np.array([rank_map[val] for val in x]),  np.array([rank_map[val] for val in y])
    else:
        xr, yr = x - np.min(x), y - np.min(y)
    return np.mean(np.abs(xr - yr)) / (len(order)  - 1 if order else 1)

def compute_distance(x: np.ndarray, y: np.ndarray, attr_type: Optional[AttributeType] = None, **kwargs) -> float:
    """have  distance be compute based on different attribute type."""
    if attr_type is None:
        attr_type = determine_attribute_type(np.asarray(x))

    if attr_type == "numeric":
        return minkowski_distance(x, y, p=kwargs.get("h", 2))
    elif attr_type == "binary":
        return binary_distance(x, y, method=kwargs.get("method", "jaccard"))
    elif attr_type == "ordinal":
        return ordinal_distance(x, y, order=kwargs.get("order"))
    elif attr_type == "nominal":
        return nominal_distance(x, y) 
    else:
        raise ValueError("Unsupported attribute type.")


Question 2

In [ ]:
import numpy as np
from typing import Optional, Literal, Union

#have attributes distance calculations
AttributeType = Literal["binary", "numeric", "nominal", "ordinal"]

def pairwise_distance_matrix(
    X: np.ndarray,
    dtype:  Optional[AttributeType] = None,
    **kwargs
) -> np.ndarray:

    """
    have the pairwise distance matrix for objects in X be computed.
    kwargs forwarded to distance comput functions (e.g., h for Minkowski).
    """
    X = np.asarray(X)
    K = X.shape[0]
    if dtype is None:
        dtype =  determine_attribute_type(X[0])

    distance_matrix  = np.zeros((K, K), dtype=float)
    for i in range(K):
        for j in range(i + 1, K):
            distance_matrix[i, j] = compute_distance(X[i], X[j], dtype=dtype, **kwargs)
            distance_matrix[j, i]  = distance_matrix[i, j]
    return distance_matrix


Question 3

In [ ]:
!pip install pandas

In [ ]:
import numpy as np
import pandas as pd
from typing import Optional, Literal

#function to compute distance matrix
def pairwise_distance_matrix(X: np.ndarray, dtype: Literal["binary", "numeric", "nominal", "ordinal"], **kwargs) -> np.ndarray:
    """
    Computes the pairwise distance matrix for an input array X.
    - X: A 2D numpy array (K x N) where K is the number of objects and N is the number of features.
    - dtype: Type of the input data (numeric, binary, nominal, ordinal).
    - kwargs: Additional arguments for distance computation.
    """
    K = X.shape[0]
    distance_matrix = np.zeros((K, K))  #distance matrix for it to be initilized 

    #Compute pairwise distances
    for i in range(K):
        for j in range(i + 1, K):  # Only have  upper  calculated (distance is symmetric)
            distance =  compute_distance(X[i], X[j], attr_type=dtype, **kwargs)
            distance_matrix[i, j] = distance_matrix[j, i]  = distance  
    
    return distance_matrix

#Function is for the distance calculations (binary, numeric, , nominal, ordinal)
def  test_distance_matrices():
    rng =  np.random.default_rng(42)
    
    #test one: for numeric data (Euclidean - Minkowski with h=2)
    X_num = rng.normal(0, 1, size=(10, 5))  # 10 objects, 5 features (numeric data)
    print("=== Numeric Distance Matrix (Euclidean - Minkowski with h=2) ===")
    D_num = pairwise_distance_matrix(X_num, dtype="numeric", h=2)  # Using Minkowski with h=2 (Euclidean)
    print(pd.DataFrame(D_num).round(3))
    
    #Test three: for nominal data (Simple Matching)
    categories = ["red", "green", "blue", "yellow"]
    X_nom = rng.choice(categories, size=(10, 5))  # 10 objects, 5 features (nominal data)
    print("=== Nominal Distance Matrix (Simple Matching) ===")
    
    D_nom = pairwise_distance_matrix(X_nom, dtype="nominal")
    print(pd.DataFrame(D_nom).round(3))
    
    #Test four: for ordinal data (Rank Normalized Manhattan)
    X_ord = rng.integers(1, 6, size=(10, 5))  # 10 objects, 5 features (ordinal data)
    print("=== Ordinal Distance Matrix (Rank Normalized Manhattan)  ===")
    
    D_ord  = pairwise_distance_matrix(X_ord, dtype="ordinal", order=[1,  2, 3, 4, 5])
    print(pd.DataFrame(D_ord).round(3))

    #test two: for binary data (Jaccard)
    X_bin = rng.integers(0, 2, size=(10, 5))  # 10 objects, 5 features (binary data)
    print("=== Binary Distance Matrix (Jaccard) ===")
    D_bin = pairwise_distance_matrix(X_bin, dtype="binary", method="jaccard")
    print(pd.DataFrame(D_bin).round(3))


if __name__ == "__main__":
    test_distance_matrices()


=== Numeric Distance Matrix (Euclidean - Minkowski with h=2) ===
       0      1      2      3      4      5      6      7      8      9
0  0.000  2.684  3.160  3.143  2.027  3.001  3.878  3.735  3.079  2.843
1  2.684  0.000  2.895  1.454  2.114  1.834  3.870  3.026  2.341  2.119
2  3.160  2.895  0.000  2.138  2.654  1.476  2.668  2.350  1.238  0.912
3  3.143  1.454  2.138  0.000  2.743  1.577  3.621  2.755  1.769  1.693
4  2.027  2.114  2.654  2.743  0.000  1.815  3.166  2.813  2.524  2.124
5  3.001  1.834  1.476  1.577  1.815  0.000  3.067  2.376  1.547  0.815
6  3.878  3.870  2.668  3.621  3.166  3.067  0.000  1.104  2.055  2.898
7  3.735  3.026  2.350  2.755  2.813  2.376  1.104  0.000  1.419  2.352
8  3.079  2.341  1.238  1.769  2.524  1.547  2.055  1.419  0.000  1.173
9  2.843  2.119  0.912  1.693  2.124  0.815  2.898  2.352  1.173  0.000
=== Nominal Distance Matrix (Simple Matching) ===
     0    1    2    3    4    5    6    7    8    9
0  0.0  0.4  1.0  0.6  0.8  1.0  0.6  0.6

Question 4

In [ ]:
import numpy as np
from typing import Sequence, Optional, Literal

#have the different attribute types be defined
AttributeType  = Literal["ordinal", "numeric", "nominal",  "binary"]

def convert_to_float(value):
    """Convert a value to float, if can be converted."""
    try:
        return float(value)
    except  Exception:
        return None

def gower_distance_matrix(
    data: np.ndarray,
    attribute_types: Sequence[AttributeType],
    ordinal_orders:  Optional[Sequence[Optional[Sequence]]] = None,
    binary_asymmetric: bool  = True,
    feature_weights: Optional[Sequence[float]] = None
) -> np.ndarray:
    """
    Compute a KxK Gower-style distance matrix for mixed attribute types.
    - data: K x N matrix where K = number of objects, N = number of features
    - attribute_types: List of N attribute types  for each feature
    
    - ordinal_orders: Optional list of explicit orders for ordinal columns
    - binary_asymmetric: If is treu , binary distance is asymmetric (Jaccard), else symmetric (Hamming)
    - feature_weights: Weights for each feature (default = 1.0 for each feat)
    """
    data = np.asarray(data, dtype=object)
    K, N = data.shape  # N = features, K = objects,

    if len(attribute_types) != N:
        raise ValueError ("attribute and their types need to match number of the columns.")
    
    if ordinal_orders is None:
        ordinal_orders =  [None] * N
    if feature_weights is None:
        feature_weights = [1.0] * N

    col_min, col_max, col_levels, ord_maps = [None]* N, [None] * N,[None] * N, [None]  * N

    for j in range(N):
        attr_type = attribute_types [j]
        col = data[:, j]

        if attr_type == "numeric ":
            colf = np.array ([convert_to_float(v) for v in col], dtype=float)
            col_min[j] = np.min(colf)
            col_max[j] = np.max(colf)
        elif attr_type == "ordinal":
            if ordinal_orders[j]:
                ord_maps[j] = {cat: idx for idx, cat in enumerate(ordinal_orders[j])}
                col_levels[j] = len(ordinal_orders[j])
            else:
                vals =  np.array([int(v) for v in col], dtype=int)
                col_min[j] = np.min(vals)
                col_max[j] = np.max(vals)
                
                col_levels[j] = col_max[j] - col_min[j] + 1

    D = np.zeros((K, K), dtype=float)
    weights =  np.array(feature_weights, dtype=float)

    for i in range(K):
        obj_i  = data[i, :]
        for k in range(i + 1, K):
            obj_k = data[k, :]

            contrib =  0.0
            wsum = 0.0
            for j in range(N):
                attr_type = attribute_types[j]
                weight = weights[j]
                if weight == 0:
                    continue

                if  attr_type == "numeric":
                    rng = float(col_max[j] - col_min[j])
                    vi = convert_to_float(obj_i[j])

                    vk = convert_to_float (obj_k[j])
                    if rng == 0 or vi is  None or vk is None:
                        dij = 0.0
                    else:
                        dij = abs(vi - vk) / rng
                elif attr_type == "ordinal":
                    if ord_maps [j]:
                        m = ord_maps[j]
                        mi = m[obj_i[j]]
                        mk = m[obj_k[j]]
                        dij = abs(mi - mk) / max(1, col_levels[j] - 1)
                    else:
                        dij = abs(int(obj_i[j]) - int(obj_k[j])) / max(1, col_levels[j] - 1)
                elif attr_type  == "nominale":
                    dij = 0.0 if obj_i[j] == obj_k[j] else 1.0
                elif attr_type == "binary":
                    vi = int(obj_i[j])
                    vk = int(obj_k[j])
                    if binary_asymmetric:
                        if vi == 0 and  vk == 0:
                            continue
                        dij = 0.0 if vi == vk else 1.0
                    else:
                        dij =  0.0 if vi == vk  else 1.0
                contrib += weight * dij
                wsum += weight

            D[i, k]  = contrib / wsum if wsum != 0  else 0.0
            D[k, i] = D[i, k]

    return D
